   * ... framing (CISC vs RISC, 3-step overview) ...
   * ... the core build-up: separate R-type datapath, separate I-type datapath,
     then merging them with a MUX...
   * extend the combined datapath: Load, Store, instruction memory, PC, branch.
     Again one idea -- "keeping adding to the same datapath".
   * ... are ALU control logic -- the truth table mapping opcode/funct to ALU
     operation.
   * ... the payoff: the full combined datapath with control unit, then 
     showing which parts "light up" for each instruction type.



   ...

   STEP 3: ADD A CONTROL UNIT TO DRIVE THE MUXes. The combined datapath can do
   everything, but it needs to be told which mode to operate in for each
   instruction. The Control Unit reads the opcode from the current instruction 
   and sets all the MUX select signals, ALU operation, and memory read/write
   signals accordingly. When it sees an R-type opcode, it configures the MUXes
   to behave like the R-type datapath from Step 1. When it sees a Load opcode, 
   it reconfigures everything to behave like the Load datapath. Same hardware, 
   different configuration each cycle. 

   The beauty of this approach is that you never need to understand the full
   combined datapath all at once. You just need to understand each simple 
   datapath individually, then see how MUXes bridge the differences, then see 
   how the control unit picks the right configuration...


   This artifact covers the "brain" of the processor -- the Control Unit and ALU
   control block ... It has three stacked sections...


---
HOW TO READ THE THREE SECTIONS
   The flow goes top -> middle -> bottom, mirroring how signals actually 
   propagate in hardware:

SECTION 1 (Instruction Encoding)
   ... shows the raw 32-bit instruction split into fields. The colour coding 
   tells you where each field goes: amber field (opcode) route to the Control
   Unit, purple fields (funct3/funct7) routes to the ALU control block, blue
   (`rs1`) and coral (`rs2`) go to the register file read ports, and green
   (`rd`) goes to the write port.

SECTION 2 (Control Unit)
   ... shows what happens when the opcode arrives. This is a pure combinational
   circuit -- no memory, no state, just a truth table baked into logic gates. 
   The 7-bit opcode goes in, and 8 control signals come out instantly. The green
   badges mean "active/on", the red ones mean "off," and dim ones are don't 
   care.

SECTION 3 (ALU Control block)
   shows the second-level decode. The ALUOp sigmnal from the Control Unit tells
   the ALU control block what class of operation this is. For Load/Store...
   it's always "add"--no need to look at funct fields. For Branch... it's
   always "subtract". Only for R-type (ALUOp=10) does the ALU control block
   actually inspect `funct3` and `funct7` to determine the exact operation.


...




   * ... recap from Lecture 6 (the single-cycle Load datapath repeated 4 times
     )
   * ... the key motivation -- the execution cycle table showing Load needs 5 
     steps but Jump only needs 2, yet single-cycle forces everything to take the
     same long cycle. This is why multi-cycle exists...

---

-- The reason we shift the branch immediate left by 1 bit is because of a clever
   hardware optimisation: in RISC-V, EVERY INSTRUCTION MUST BE AT LEAST 2-BYTE
   ALIGNED. This means every instruction address in memory is an even number 
   (ending in a binary `0`). Since we know the last bit of the target address 
   will always be zero, there is no reason to waste one of our precious 12 bits
   in the instruction format to store it. 

   By "ignoring" that final zero and shifting everything left when we 
   reconstruct the address, we effectively double our branching range. Instead 
   of a 12-bit offset allowing us to jump +-2 KB, the shift allows that same
   12-bit field to reach +-4 KB. In the hardware datapath, this is handled by a 
   simple SHIFT LEFT 1 unit (basically just moving wires over by opne pin) 
   before the immediate is added to the Program Counter (PC).

   

-- The R-TYPE DATAPATH is the circuity used for arithmetic and logical 
   operations involving only registers. When an instruction like `add x1, x2, x3`
   is fetched, the Program Counter (PC) provides the address to Instruction 
   Memory. The "register file" then reads the value from the two source 
   registers (`rs1=x2` and `rs2=x3`) and sends them to the ALU (Arithmetic Logic
   Unit). The ALU performs the operation (addition) based on the control signals 
   derived from the `funct7` and `funct3` fields. Finally, the result is wired 
   back from the ALU output and written into the destination register (`rd`)
   (`x1`) during the "Write Back" stage.

   An I-TYPE DATAPATH is slightly more complex because it involves an 
   "IMMEDIATE" value--a constant number hardcoded into the instruction itself. 
   For an instruction like `addi x5, x6, x10` (add immediate), the datapath
   fetches the value of `x6` from the register file, but the second input to the
   ALU isn't another register. Instead, the 12-bit immediate field from the
   instruction is send through a SIGN-EXTEND unit to turn it into a 32-bit
   number (10). A multiplexer (MUX) tells the ALU to ignore the second register
   output and use this extended immediate value instead. The sum is then stored
   in the destination register (`rd=x5`).

   The I-type datapath is also the foundation for LOAD INSTRUCTIONS, such as 
   `lw x10, 4(x11)`. In this "Load example," the ALU doesn't calculate a final
   sum to store in a register; instead, it calculates a MEMORY ADDRESS by adding
   the base address in `x11` to the offset of 4. The data path then takes this
   calculated address and uses it to "point" into DATA MEMORY (RAM). The 32-bit
   word located at that address is retrieved and then sent back to the register
   file to be saved in `rd=x10`. This requires an extra step (Memory Access) 
   that R-type instructions completely skip.

   





-- The RISC-V register file is a set of 32 ultra-fast, small-capacity memory
   locations within the cpu, USED FOR TEMPORARY DATA STORAGE, operand retrieval,
   and holding calculation results during instruction execution. It typically
   consists of 32 general-purpose registers (x0 - x31), where `x0` is hardwired
   to zero, allowing efficient data manipulation. 

   Think of `lw` (Load Word) as the "bridge" that brings data from your slow,
   massive RAM into your lightning-fast CPU registers so you can actually do 
   math with it.


HOW `lw` WORKS
   Since RISC-V is a LOAD-STORE ARCHITECTURE, the ALU cannot talk to memory
   directly. If you want to add two numbers stored in RAM, you must first "load"
   them into registers.

   The `lw` instruction uses the I-type (Immediate) format: 
   `lw rd, offset(rs1)`
      1. BASE ADDRESS: It looks at the address stored in `rs1` (the base).
      2. OFFSET: It adds a constant number (the offset) to that base to find
         the exact memory location.
      3. THE MOVE: It copies 32 bits (1 word) from that memory address into the
         destination register (`rd`).


---
EXAMPLE: Loading an Array Element
   Imagine you have an array of integers in C: `int initial_score = scores[2];`

   In memory, `scores` starts at address `0x4000`. Since each `int` is 4 bytes,
   `scores[2]` is at `0x4008` ...


THE RISC-V ASSEMBLY:
   Assume `x10` already holds the base address `0x4000`.

```Assembly
lw x5, 8(x10)   # x5 = Memory[x10 + 8]
```
   * `rs1 (x10)`: The base pointer to the start of the array (`0x4000`)
   * `offset (8)`: We jump 8 bytes ahead to get to the 3rd element.
   * `rd (x5)`: The "score" is now ...


---
KEY CONCEPT: Byte Addressing
   ... memory is BYTE-ADDRESSED. Even though we load a 32-bit "word", the 
   addresses move by 1-byte incremeents. This is why to get to the next "word"
   in an array, you always add `4` to the offset (4 bytes = 32 bits).
   
      







-- In RISC-V... a "word" is defined as 32 BITS (4 bytes), regardless of whether 
   the processor is 32-bit (RV32), 64-bit (RV64), or 128-bit. It is used to 
   represent data, memory addresses, or instructions. Assembly instructions like
   `lw` (load word) and `sw` (store word) operate on this 32-bit width.

   Key details...:
   * WORD: 32 bits (4 bytes)
   * Halfword: 16 bits (2 bytes)
   * Doubleword: 64 bits (8 bytes)
   * Double Word/Quadword: 128 bits (16 bytes)
   * Memory Addressing: RISC-V is byte-addressable, meaning each 32-bit word
     occupies four unique consecutive addresses (e.g., a word at address 4 uses
     bytes 4, 5, 6, and 7).
   * ASSEMBLY DIRECTIVE: The `.word` assembly directive is used to allocate 
     space for or insert 32-bit data values.
   * LOAD/STORE VARIANTS: `lb/sb` (byte), `lh/sh` (half word), and `lw`/`sw`
     (word).  


   At the most basic level, RISC (Reduced Instruction Set Computer), like the 
   RISC-V ... is built on the philosophy that "simple is fast". Every 
   instruction is the same size (32 bits) and usually takes just one clock cycle
   to execute. By keeping the instruction basic--like only having simple `add`,#
   `sub`, or `lw`--the hardware can be streamlined to run at very high clock
   speeds. The trade-off is that you need more lines of code to do complex tasks
   ; for eaxmple, a single math operation might require several separate RISC
   instruction to load data, process it, and store back.

   CISC (Complex Instruction Set Computer), like the Intel x86 chips in most 
   laptops, takes the opposite approach: "make the hardware do the heavy 
   lifting." CISC instructions vary in length and can perform complex multi-step
   operations in a single command (like "multiply these two numbers directly in
   memory and store the result"). This makes the compiled code much shorter and
   saves memory, which was crucial back when RAM was incredibly expensive. 
   However, this makes the hardware "decoder" inside the CPU much bulkier and 
   more power-hungry because it has to figure out how to handle all those 
   different instruction shapes.

   A great analogy is COOKING: RISC is like having a kitchen with only basic
   ...

---

---

WHY SINGLE-CYCLE HAS A FUNDAMENTAL PROBLEM
   In the single-cycle datapath you just learned, every instruction -- no matter
   how simple -- completes in exactly one clock cycle. That sounds clean, but
   here's the catch: the clock period must be long enough to accomodate the
   SLOWEST instruction. And the slowest instruction is `ld` (load), because it
   touches every single component in the datapath:

   1. INSTRUCTION FETCH -- read from instruction memory
   2. READ REGISTER -- read `rs1` from the register file
   3. ALU -- compute the memory address (base + offset)
   4. MEMORY -- read from data memory at that address
   5. WRITE REGISTER -- write the loaded value back to `rd`


   That's five sequential stages, and the clock can't tick until all five 
   finishes. Now consider `add x9, x20, x21` -- it only needs stages 1, 2, 3,
   and 5 (no memory access). It could finish faster, but it's forced to wait for
   the full load-length cycle. Even worse, a jump insertion (J-type) only needs
   stage 1 and 3 -- fetch and compute the new PC -- but it still wastes the same
   amount of time sitting idle.

   ...

   In single-cycle, the clock period = time for 5 stages (the load path). Every
   instruction pays that cost, even jump which only needs 2 stages. If each 
   stage takes 200ps, the clock period is 1000ps regardless. A jump instruction
   wastes 600ps doing nothing.


THE MULTI-CYCLE IDEA: ONE STAGE PER CLOCK CYCLE
   The solution is ... simple: instead of doing everything in one long cycle,
   break each instruction into multiple short cycles, where each cycle does
   exactly ONE STAGE OF WORK. Now the clock period only needs to be as long as
   the slowest single stage (say 200ps), not the slowest instruction.

   Load takes 5 short cycles (5 \times 200ps = 1000ps -- same total time). But
   `add` takes 4 cycles... On average, instructions finish faster because 
   simple instructions don't pay the load tax.


---
WHAT CHANGES ARCHITECTURALLY
   This seemingly simple idea has three major consequences for the hardware:


1. YOU NEED INTERNAL REGISTERS TO HOLD INTERMEDIATE RESULTS BETWEEN CYCLES.
   In single-cycle, data flows from the instruction memory through the register
   file through the ALU through data memory all in one combinational wave. In
   multi-cycle, the ALU computes the address in cycle 3, but memory doesn't read
   until cycle 4 -- so the ALU result needs to be stored somewhere between those
   cycles. That's why the multi-cycle datapath introduces new registers you 
   didn't see before: IR (Instruction Register -- holds the fetched instruction)
   , A and B (hold register values read in cycle 2), ALUOut (holds the ALU
   result from cycle 3), and MDR (Memory Data Register -- holds data read from
   memory in cycle 4). These are the "local variables" that carry data between
   cycles.


2. YOU CAN REUSE HARDWARE ACROSS CYCLES. In single-cycle, you need separate 
   instruction memory and data memory because both are access in the same cycle.
   In multi-cycle, instruction fetch (cycle 1) and data access (cycle 4) happen
   in different cycles, so they can share ONE MEMORY -- it's just used for 
   instructions in cycle 1 and data in cycle 4. Similarly, the ALU is used for
   PC+4 in cycle 1, address computation in cycle 3, and the actual operation in
   cycle 3 for R-type. One ALU, reused multiple times. Less silicon.


3. THE CONTROL UNIT BECOMES A STATE MACHINE, NOT A SIMPLE COMBINATORIAL CIRCUIT. 
   In single-cycle, the control unit is purely combinational -- opcode in, 
   signal out, done in one cycle. In multi-cycle, the control unit needs to
   remember which cycle of which instruction it's currently in. That's state.
   So the control unit becomes a FINITE STATE MACHINE (FSM) with 11 states 
   (states 0-10), where each state outputs a specific set of control signals and
   transitions to the next state based on the opcode. The FSM loops: after an
   instruction finishes (however many cycles it needs), it returns to state 0
   to fetch the next instruction.


THE 5 STAGES BY NAME
   ... standard abbreviations you'll see everywhere in computer architecture:

   IF (Instruction Fetch) -- `IR = M[PC]`, `PC = PC + 4`. Fetch the instruction
   and increment PC. Every instruction starts here.

   ID (Instruction Decode) -- `A = R[rs1]`, `B = R[rs2]`. Read the registers. 
   Also every instruction does this (even if it won't use B -- it's speculative,
   and it's free since the register file is fast).

   EX (Execute) -- This is where paths diverge. R-type does `ALUOut = A op B`.
   Load/Store does `ALUOut = A + imm` (address calculation). Branch checks the
   condition and potentially updates PC. 

   MEM (Memory Access) -- Only Load and Store reach here. Load does 
   `MDR = M[ALUout]`. Store does `M[ALUOut] = B`.

   WB (Write Back) -- R-type does `R[rd] = ALUOut`. Load does `R[rd] = MDR`.
   Branch and Store skip this entirely.

   The key insight... you can prove correctness by substituting backwards. For
   load: `R[rd] = MDR` (cycle 5), and `MDR = M[ALUOut]` (cycle 4), and 
   `ALUOut = A + imm` (cycle 3), and `A = R[rs1]` (Cycle 2). Chain them:
   `R[rd] = M[R[rs1] + imm]`. That's exactly the load instruction's definition.
   The five small steps compose into the correct overall effect. 

#### The FSM control unit
   The state diagram... the complete "brain" of the multi-cycle processor. It 
   has 11 states numbered 0-10. Every instruction starts at state 0 (IF) and
   state 1 (ID). Then the opcode determines which path to take: 

            R-type: state 0 → 1 → 2 → 4 (4 cycles)
            I-type: state 0 → 1 → 3 → 4 (4 cycles)
            Load: state 0 → 1 → 5 → 6 → 7 (5 cycles)
            Store: state 0 → 1 → 5 → 8 (4 cycles)
            Branch: state 0 → 1 → 9 (3 cycles)
            Jump: state 0 → 1 → 10 (3 cycles)


   After each instruction finishes at its terminal state, the FSM jumps back to
   state 0 to fetch the next instruction. Each state specifies exactly which 
   control signals are asserted -- for example, state 0 asserts PCWrite, IRWrite
   , MemRead (to fetch the instruction and update PC), while 

---

-- ... the MULTI-CYCLE DATAPATH. 